# STgram-MFN Colab Training Launcher

This notebook is the Colab entry point for the STgram-MFN / pretrained-checkpoint experiment. The real training logic lives in `pipeline/05_train_stgram.py` and evaluation lives in `pipeline/06_evaluate_stgram.py`, so the notebook stays small and the project remains reusable from scripts, GitHub, and Colab.

Use this notebook for GPU training. Do not run full training locally.

## 1. Enable GPU

In Colab, go to `Runtime > Change runtime type > T4 GPU`, then run the next cell.

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. Clone The Repo

This checks out the active migration branch.

In [ ]:
!git clone https://github.com/SARWAGYASHAH/Anomalous-Sound-Detection-using-Spectrograms.git
%cd Anomalous-Sound-Detection-using-Spectrograms
!git checkout migrate-to-keras
!git pull

## 3. Install Dependencies

In [ ]:
# Keep Colab's preinstalled GPU stacks intact.
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps

import joblib, librosa, sklearn
print('Dependencies ready')

## 4. Attach Data

The raw gearbox zip is stored on Google Drive and downloaded with `gdown`.

If the download cell fails, open the Drive file sharing settings and set access to `Anyone with the link`.

In [ ]:
from pathlib import Path

DRIVE_ZIP_URL = 'https://drive.google.com/file/d/1p6TDo1GpTWHQzfHRcxs7NgXtG4-xVzqS/view?usp=drive_link'
ZIP_PATH = Path('Data/dev_data_gearbox.zip')
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    !gdown --fuzzy "$DRIVE_ZIP_URL" -O Data/dev_data_gearbox.zip

print('Zip exists:', ZIP_PATH.exists())
print('Zip size MB:', round(ZIP_PATH.stat().st_size / (1024 * 1024), 2) if ZIP_PATH.exists() else 'missing')

In [ ]:
!unzip -q -o Data/dev_data_gearbox.zip -d Data/
!ls Data
!find Data/gearbox -maxdepth 2 -type d | sort

## 5. Verify Raw Data

STgram-MFN trains directly from raw `.wav` files, so this notebook does not need the spectrogram preprocessing step used by the Keras autoencoder notebook.

In [ ]:
from pathlib import Path

for path in [
    Path('Data/gearbox/train'),
    Path('Data/gearbox/source_test'),
    Path('Data/gearbox/target_test'),
]:
    files = list(path.glob('*.wav')) if path.exists() else []
    normal = sum('normal' in file.name.lower() for file in files)
    anomaly = sum('anomaly' in file.name.lower() for file in files)
    print(path, 'total=', len(files), 'normal=', normal, 'anomaly=', anomaly)

## 6. Optional Pretrained Checkpoint

Leave `PRETRAINED_CHECKPOINT` empty for the first run. Later, you can point it to a previous `best_model.pt` in Drive and optionally freeze the backbone for a lighter fine-tuning run.

In [ ]:
PRETRAINED_CHECKPOINT = ''
FREEZE_BACKBONE = False

extra_args = ''
if PRETRAINED_CHECKPOINT:
    extra_args += f' --pretrained-checkpoint "{PRETRAINED_CHECKPOINT}"'
if FREEZE_BACKBONE:
    extra_args += ' --freeze-backbone'

print('Extra training args:', extra_args or '(none)')

## 7. Dry Run

This builds the raw-audio dataset/model and runs one forward pass. It does not train.

In [ ]:
!python pipeline/05_train_stgram.py --config config/stgram_mfn.yaml --dry-run --batch-size 4 --num-workers 2 $extra_args

## 8. First Short Training Run

Start with a short run to verify Colab paths, GPU, model saving, GMM fitting, and graph generation.

In [ ]:
!python pipeline/05_train_stgram.py --config config/stgram_mfn.yaml --epochs 3 --batch-size 32 --num-workers 2 $extra_args

## 9. Full Training Run

Run this after the short run succeeds. Every run creates a new versioned directory under `artifacts/models/stgram_mfn/`, so previous trained models are not overwritten.

In [ ]:
!python pipeline/05_train_stgram.py --config config/stgram_mfn.yaml $extra_args

## 10. Evaluate Source And Target Test Sets

In [ ]:
!python pipeline/06_evaluate_stgram.py --config config/stgram_mfn.yaml --split source_test --assets-dir docs/assets
!python pipeline/06_evaluate_stgram.py --config config/stgram_mfn.yaml --split target_test --assets-dir docs/assets

## 11. Inspect Metrics And Graphs

In [ ]:
import json
from pathlib import Path

for split in ['source_test', 'target_test']:
    metrics_path = Path('artifacts/evaluation_stgram') / split / 'metrics.json'
    metrics = json.loads(metrics_path.read_text())
    print('\n' + split)
    print(json.dumps(metrics['metrics'], indent=2))

In [ ]:
from IPython.display import Image, display
from pathlib import Path

latest_run = Path(Path('artifacts/models/stgram_mfn/latest.txt').read_text().strip())
display(Image(filename=str(latest_run / 'training_loss.png')))

for path in [
    Path('artifacts/evaluation_stgram/source_test/roc_curve.png'),
    Path('artifacts/evaluation_stgram/source_test/anomaly_score_distribution.png'),
    Path('artifacts/evaluation_stgram/target_test/roc_curve.png'),
    Path('artifacts/evaluation_stgram/target_test/anomaly_score_distribution.png'),
]:
    display(Image(filename=str(path)))

## 12. Save Artifacts To Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn
!cp -r artifacts/models/stgram_mfn /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn/models
!cp -r artifacts/evaluation_stgram /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn/evaluation
!cp -r docs/assets /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn/docs_assets
!find /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn -maxdepth 3 -type f | sort | tail -80